### RAG Pipelines- Data Ingestion to Vector DB Pipelines

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: 30072026_PG_Spot1_MinAllScore.pdf


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)


  ✓ Loaded 23 pages

Total documents loaded: 23


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 26.5.2 (Build 25F84) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260730120613Z00'00'", 'moddate': "D:20260730120613Z00'00'", 'source': '..\\data\\text_files\\pdf\\30072026_PG_Spot1_MinAllScore.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': '30072026_PG_Spot1_MinAllScore.pdf', 'file_type': 'pdf'}, page_content='UNIVERSITY OF DELHI  ADMISSION BRANCH POSTGRADUATE ADMISSIONS 2026-27 FIRST SPOT ROUND OF ALLOCATION AND ADMISSIONS - MINIMUM ALLOCATION SCORE*  \n *Disclaimer:  1. Allocations are based on the following criteria: programme-specific eligibility, merit, social category, availability of seats, tie-breaking rules.  2. The Minimum allocation scores are shown only for programs where seats were allocated in Spot Round I. 3. The University strives for accuracy. However, applicants are encouraged to report any discrepancies to the Admission Branch, University of Delhi, for timely resolution.  4. The fi

### chunking

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 23 documents into 90 chunks

Example chunk:
Content: UNIVERSITY OF DELHI  ADMISSION BRANCH POSTGRADUATE ADMISSIONS 2026-27 FIRST SPOT ROUND OF ALLOCATION AND ADMISSIONS - MINIMUM ALLOCATION SCORE*  
 *Disclaimer:  1. Allocations are based on the followi...
Metadata: {'producer': 'macOS Version 26.5.2 (Build 25F84) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260730120613Z00'00'", 'moddate': "D:20260730120613Z00'00'", 'source': '..\\data\\text_files\\pdf\\30072026_PG_Spot1_MinAllScore.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': '30072026_PG_Spot1_MinAllScore.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 26.5.2 (Build 25F84) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260730120613Z00'00'", 'moddate': "D:20260730120613Z00'00'", 'source': '..\\data\\text_files\\pdf\\30072026_PG_Spot1_MinAllScore.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': '30072026_PG_Spot1_MinAllScore.pdf', 'file_type': 'pdf'}, page_content='UNIVERSITY OF DELHI  ADMISSION BRANCH POSTGRADUATE ADMISSIONS 2026-27 FIRST SPOT ROUND OF ALLOCATION AND ADMISSIONS - MINIMUM ALLOCATION SCORE*  \n *Disclaimer:  1. Allocations are based on the following criteria: programme-specific eligibility, merit, social category, availability of seats, tie-breaking rules.  2. The Minimum allocation scores are shown only for programs where seats were allocated in Spot Round I. 3. The University strives for accuracy. However, applicants are encouraged to report any discrepancies to the Admission Branch, University of Delhi, for timely resolution.  4. The fi

### Embeddings

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\Coding\AI Engineering\Agentic AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6535.46it/s]


Model loaded successfully. Embedding dimension: 384
